In [1]:
import pandas as pd
import numpy as np

from loader import cargar_dataset_limpio

### Carga del Dataset

In [2]:
df = cargar_dataset_limpio()

In [3]:
display(df)

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,tmax,velmedia,...,dir,racha,horaracha,hrMax,horaHrMax,hrMin,horaHrMin,lon,lat,dir_tipo
0,1970-01-01,C249I,FUERTEVENTURA AEROPUERTO,LAS PALMAS,25,19.8,0.0,16.0,23.5,6.7,...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,NaN,-13.863056,28.444722,Válida
1,1970-01-01,1679A,MONFORTE DE LEMOS,LUGO,291,4.0,0.0,0.0,8.0,<NA>,...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,NaN,-7.510833,42.531667,Válida
2,1970-01-01,2462,PUERTO DE NAVACERRADA,MADRID,1893,-5.0,0.4,-7.0,-3.0,3.1,...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,NaN,-4.010556,40.793056,Válida
3,1970-01-01,1212E,ASTURIAS AEROPUERTO,ASTURIAS,127,4.3,2.6,2.0,6.6,0.0,...,99.0,7.2,00:13,<NA>,NaN,<NA>,NaN,-6.044167,43.566944,Varias
4,1970-01-01,0016A,REUS AEROPUERTO,TARRAGONA,71,5.5,0.0,0.6,10.4,1.7,...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,NaN,1.163611,41.145,Válida
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7424496,2024-12-31,9562X,MORELLA,CASTELLON,990,4.8,0.0,0.8,8.8,1.4,...,12.0,5.0,16:50,97.0,23:50,63.0,14:00,-0.101944,40.621667,Válida
7424497,2024-12-31,1002Y,"BAZTAN, IRURITA",NAVARRA,183,4.2,0.0,-4.9,13.4,0.0,...,22.0,3.1,16:20,100.0,09:00,50.0,14:10,-1.543056,43.135833,Válida
7424498,2024-12-31,3254Y,MORA,TOLEDO,717,2.0,0.0,-3.1,7.1,1.1,...,20.0,3.9,14:30,100.0,09:20,80.0,11:20,-3.780556,39.686944,Válida
7424499,2024-12-31,0194D,"CORBERA, PUIG D'AGULLES",BARCELONA,647,<NA>,0.0,<NA>,<NA>,<NA>,...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,NaN,1.885,41.408056,Válida


In [4]:
df.dtypes

fecha          datetime64[ns]
indicativo     string[python]
nombre         string[python]
provincia      string[python]
altitud                 Int64
tmed                  Float64
prec                  Float64
tmin                  Float64
tmax                  Float64
velmedia              Float64
sol                   Float64
presMax               Float64
horaPresMax            object
presMin               Float64
horaPresMin            object
hrMedia               Float64
horatmin               object
horatmax               object
dir                   Float64
racha                 Float64
horaracha              object
hrMax                 Float64
horaHrMax              object
hrMin                 Float64
horaHrMin              object
lon                   Float64
lat                   Float64
dir_tipo               object
dtype: object

### Reestructuración en formato tidy data

In [5]:
# Columna con las fechas
COL_FECHA = 'fecha'

# Columnas asociadas a la estación de medición
cols_estacion = [
    'indicativo',
    'nombre',
    'provincia',
    'altitud',
    'lon',
    'lat'
]
# Pares variable - hora
dict_horas = {
    'tmin': 'horatmin',
    'tmax': 'horatmax',
    # 'presMax': 'horaPresMax',
    # 'presMin': 'horaPresMin',
    'racha': 'horaracha',
    'hrMax': 'horaHrMax',
    'hrMin': 'horaHrMin'
}

# Variables sin hora asociada
cols_var_sin_horas = [
    'tmed',
    'prec',
    'velmedia',
    'sol',
    'hrMedia',
    'dir'
    # 'dir_tipo'
]


In [6]:
# Nombres de las columnas a añadir
COL_VAR = 'variable'
COL_VAL = 'valor'
COL_HORA = 'hora'
COL_TIPO = 'tipo_hora'

In [7]:
# Despivotamos las columnas sin hora, añadimos columna 'hora' y la llenamos de NAs
df_sin_horas = df.melt(
    id_vars = [COL_FECHA] + cols_estacion,
    value_vars = cols_var_sin_horas,
    var_name = COL_VAR,
    value_name = COL_VAL
)

df_sin_horas[COL_HORA] = pd.Series([pd.NaT] * len(df_sin_horas), dtype = 'datetime64[ns]').dt.time

# df_sin_horas['hora'] = pd.NA
df_sin_horas[COL_TIPO] = pd.NA

# Pasamos a time para que los deje como NaT y ser consistente con las horas cuando se concatene
# df_sin_horas[COL_HORA] = pd.to_datetime(df_sin_horas[COL_HORA], format = '%H:%M', errors = 'coerce').dt.time

In [8]:
# Troceamos el Dataframe en cuartetos de columnas: 'fecha', 'variable', 'valor' y 'hora'
lista_dfs_temp = []
for variable, col_hora in dict_horas.items():
    df_temp = df[[COL_FECHA] + cols_estacion + [variable, col_hora]].rename(columns = {variable: COL_VAL, col_hora: COL_HORA}) # La columna con el nombre de la variable, que contiene los valores, pasa a llamarse 'valor'
    df_temp[COL_VAR] = variable
    lista_dfs_temp.append(df_temp)

df_con_horas = pd.concat(lista_dfs_temp, ignore_index=True)

In [9]:
# Procesamiento de horas
CATEGORIAS = ['Desconocida',
              'Varias',
              'Válida',
              'Inválida'] # Valor por defecto si no cuadra en ninguna de las demás

filtro_formato = df_con_horas[COL_HORA].str.match(r'^\d{1,2}:\d{2}$', na=False) # Comprueba que formato HH:MM
df_hhmm = df_con_horas.loc[filtro_formato, COL_HORA].str.split(":", expand=True).astype(int) # Df con dos columnas: HH y MM
filtro_rango = (df_hhmm[0].between(0, 23)) & (df_hhmm[1].between(0, 59)) # Comprueba HH entre 0-23; MM entre 0-59 
filtro_hhmm = filtro_rango.reindex(df_con_horas.index, fill_value = False) # Para mismo tamaño que original

# Anotamos anomalías en la hora
filtros = [
    df_con_horas[COL_HORA].isna(),
    df_con_horas[COL_HORA] == 'Varias',
    filtro_hhmm
]

df_con_horas[COL_TIPO] = np.select(filtros, CATEGORIAS[:-1], default = CATEGORIAS[-1])
df_con_horas[COL_TIPO] = df_con_horas[COL_TIPO].astype('category')

# Pasamos las horas a tipo time
df_con_horas[COL_HORA] = pd.to_datetime(
    df_con_horas[COL_HORA], format = '%H:%M', errors = 'coerce'
).dt.time

# Creamos columna datetime respectiva
# df_prep['fecha' + col] = pd.to_datetime(
#     df_prep[COL_FECHA].dt.date.astype(str) + ' ' + df_prep[col].astype(str),
#     format = '%Y-%m-%d %H:%M',
#     errors = 'coerce'
# )
    

In [10]:
df_tidy = pd.concat([df_sin_horas, df_con_horas], ignore_index=True)
# df_prep[COL_HORA] = df_prep[COL_HORA].dt.time

C:\Users\Big Data\AppData\Local\Temp\ipykernel_41992\4064210335.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_tidy = pd.concat([df_sin_horas, df_con_horas], ignore_index=True)


In [11]:
display(df_tidy.sample(20))

,fecha,indicativo,nombre,provincia,altitud,lon,lat,variable,valor,hora,tipo_hora
67388329,1982-12-02,C249I,FUERTEVENTURA AEROPUERTO,LAS PALMAS,25,-13.863056,28.444722,hrMax,<NA>,NaT,Desconocida
64514120,2017-09-13,8309X,UTIEL,VALENCIA,758,-1.244722,39.575556,racha,7.8,17:10:00,Válida
29470918,2024-04-18,6057X,MANILVA,MALAGA,140,-5.26,36.378333,sol,<NA>,NaN,NaN
44906284,1978-05-26,3013,MOLINA DE ARAGÓN,GUADALAJARA,1062,-1.878889,40.841667,tmin,6.2,07:15:00,Válida
17366207,2008-01-20,C329B,"LA GOMERA, AEROPUERTO",SANTA CRUZ DE TENERIFE,219,-17.210833,28.031667,velmedia,5.3,NaN,NaN
41324657,2014-09-16,C458U,CITFAGRO_71_PALOB,SANTA CRUZ DE TENERIFE,595,-16.578056,28.368611,dir,<NA>,NaN,NaN
35091696,2018-08-04,C456E,CITFAGRO_10_RAVELO,SANTA CRUZ DE TENERIFE,922,-16.408611,28.453889,hrMedia,<NA>,NaN,NaN
3241881,2011-04-30,9228J,OROZ-BETELU/OROTZ-BETELU,NAVARRA,635,-1.300278,42.896111,tmed,12.6,NaN,NaN
51495628,2023-07-13,2400E,AUTILLA DEL PINO,PALENCIA,874,-4.602778,41.995556,tmin,12.7,05:00:00,Válida
17816998,2010-04-14,1109,SANTANDER AEROPUERTO,CANTABRIA,3,-3.825556,43.423889,velmedia,<NA>,NaN,NaN


In [12]:
# df_tidy.to_csv('Valores_Climatologicos_1970_2024_Tidy.csv', sep = ';', index = False)
# df_tidy.drop(columns = [COL_TIPO]).to_csv('Valores_Climatologicos_1970_2024_Tidy.csv', sep = ';', index = False)
# df_tidy.drop(columns = ['indicativo', 'altitud', 'lon', 'lat', 'hora', 'tipo_hora']).to_csv('Valores_Climatologicos_1970_2024_Tidy_Lite.csv', sep = ';', index = False)
COLS_ESENCIALES = [
    'fecha',
    'nombre',
    'provincia'
]
df_tidy.to_csv('Valores_Climatologicos_1970_2024_Tidy_Lite.csv', sep = ';', index = False, columns = COLS_ESENCIALES + [COL_VAR, COL_VAL])

In [13]:
raise Exception()

Exception: 

In [ ]:
display(df_tidy.drop(columns = [COL_TIPO, 'indicativo', 'altitud', 'lon', 'lat', 'hora']))

In [ ]:
df_tidy.sample(20)

In [ ]:
df_sin_horas.sample(20)

In [ ]:
# De aquí para abajo es en sucio, ignorar

In [ ]:
# --- Clasificación de tipo_hora ---
filtro_formato = df["hora"].str.match(r'^\d{1,2}:\d{2}$', na=False)

# split HH:MM solo para las válidas
df_hhmm = df.loc[filtro_formato, "hora"].str.split(":", expand=True).astype(int)
filtro_rango = (df_hhmm[0].between(0, 23)) & (df_hhmm[1].between(0, 59))

# Alinear tamaños (True/False por fila del df original)
filtro_hhmm = filtro_rango.reindex(df.index, fill_value=False)

filtros = [
    df["hora"].isna(),
    df["hora"] == "Varias",
    filtro_hhmm
]

df["tipo_hora"] = np.select(filtros, CATEGORIAS[:-1], default=CATEGORIAS[-1])
df["tipo_hora"] = df["tipo_hora"].astype("category")

# --- Convertir a tipo time ---
df["hora"] = pd.to_datetime(
    df["hora"], format="%H:%M", errors="coerce"
).dt.time

# --- Crear fecha+hora ---
df["fechahora"] = pd.to_datetime(
    df["fecha"].dt.date.astype(str) + " " + df["hora"].astype(str),
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

In [ ]:
# Columnas asociadas a la estación de medición
cols_estacion = [
    'indicativo',
    'nombre',
    'provincia',
    'altitud',
    'lon',
    'lat'
]

# id_vars = ["fecha", "indicativo", "nombre", "provincia", "altitud", "lon", "lat"]

# Pares variable - hora
dict_horas = {
    "tmin": "horatmin",
    "tmax": "horatmax",
    "presMax": "horaPresMax",
    "presMin": "horaPresMin",
    "racha": "horaracha",
    "hrMax": "horaHrMax",
    "hrMin": "horaHrMin"
}

# Variables sin hora
cols = [
    "tmed", "prec", "velmedia", "sol", "hrMedia", "dir"
]

# --- 1) Melt para las que NO tienen hora ---
df_nohora = df.melt(
    id_vars=id_vars,
    value_vars=without_hour,
    var_name="variable",
    value_name="valor"
)
df_nohora["hora"] = pd.NA  # sin hora asociada

# --- 2) Melt para las que SÍ tienen hora ---
dfs = []
for var, hora_col in with_hour.items():
    tmp = df[id_vars + [var, hora_col]].rename(columns={var: "valor", hora_col: "hora"})
    tmp["variable"] = var
    dfs.append(tmp)

df_hora = pd.concat(dfs, ignore_index=True)

# --- 3) Unimos todo ---
df_tidy = pd.concat([df_nohora, df_hora], ignore_index=True)

# Opcional: ordenar
df_tidy = df_tidy.sort_values(["fecha", "provincia", "variable"]).reset_index(drop=True)


In [ ]:
cols_horas = [
    'horaPresMax',
    'horaPresMin',
    'horatmin',
    'horatmax',
    'horaracha',
    'horaHrMax',
    'horaHrMin'
]

df_prep[cols_horas].astype(str).values

# sorted(list(set(df_prep[cols_horas].astype(str).values)))[-5:]

lista_vals_horas = df_prep[cols_horas].astype(str).values

lista_vals_horas_flat = [val for sublista in lista_vals_horas for val in sublista]

In [ ]:
# Añadimos el sufijo :00 a las columnas cuyos valores carecen de él
cols_sufijo = ['horaPresMax', 'horaPresMin']
df_prep[cols_sufijo] = df_prep[cols_sufijo].where(
    df_prep[cols_sufijo].isna() | (df_prep[cols_sufijo] == 'Varias'),
    df_prep[cols_sufijo] + ':00'
)

In [ ]:
# Procesamiento de horas
CATEGORIAS = ['Desconocida',
              'Varias',
              'Válida',
              'Inválida'] # Valor por defecto si no cuadra en ninguna de las demás

for col in cols_horas:

    filtro_formato = df_prep[col].str.match(r'^\d{1,2}:\d{2}$', na=False) # Comprueba que formato HH:MM
    df_hhmm = df_prep.loc[filtro_formato, col].str.split(":", expand=True).astype(int) # Df con dos columnas: HH y MM
    filtro_rango = (df_hhmm[0].between(0, 23)) & (df_hhmm[1].between(0, 59)) # Comprueba HH entre 0-23; MM entre 0-59 
    filtro_hhmm = filtro_rango.reindex(df_prep.index, fill_value = False) # Para mismo tamaño que original
    
    # Anotamos anomalías en la hora
    filtros = [
        df_prep[col].isna(),
        df_prep[col] == 'Varias',
        filtro_hhmm
    ]

    df_prep[col + '_tipo'] = np.select(filtros, CATEGORIAS[:-1], default = CATEGORIAS[-1])
    df_prep[col + '_tipo'] = df_prep[col + '_tipo'].astype('category')

    # Pasamos las horas a tipo time
    df_prep[col] = pd.to_datetime(
        df_prep[col], format = '%H:%M', errors = 'coerce'
    ).dt.time

    # Creamos columna datetime respectiva
    df_prep['fecha' + col] = pd.to_datetime(
        df_prep[COL_FECHA].dt.date.astype(str) + ' ' + df_prep[col].astype(str),
        format = '%Y-%m-%d %H:%M',
        errors = 'coerce'
    )
    

In [ ]:
# Procesamos dirección del viento
dict_dir = {
    88: 'Desconocida',
    99: 'Varias'
}
dir_por_defecto = 'Válida'

df_prep['dir_tipo'] = df_prep['dir'].map(dict_dir).fillna(dir_por_defecto)

In [ ]:
df_prep.loc[df_prep['provincia'] == 'BALEARES', 'provincia'] = 'ILLES BALEARS'
df_prep.loc[df_prep['provincia'] == 'SANTA CRUZ DE TENERIFE', 'provincia'] = 'STA. CRUZ DE TENERIFE'

In [ ]:
df_prep.loc[
    (df_prep['indicativo'] == '6084X') &
    (df_prep['tmin'] == 50.0) &
    (df_prep['tmax'] == -50.0),
    ['tmax', 'tmin']
] = np.nan

In [ ]:
df_prep.to_csv('Valores_Climatologicos_1970_2024_Limpios.csv', sep = ';', index = False)